# Requirement
## Setting up environment
Please install `openai`, `tiktoken`,`mistletoe`, `marker-pdf`,and `pandas` along with generating an API Key for interacting with OpenAI platform
```
pip install openai tiktoken mistletoe pandas marker-pdf
```
## Setting up environment variables
Create a file at the working directory named `env.json`
Edit the file content with the following:
```
{
    "OPENAI_API_KEY": "<Insert your openai api key here>"
}
```

In [5]:
import subprocess as sp
import os
import json
from utils import scoring_markdown_annual_report
from openai import OpenAI
import pandas as pd

In [6]:
# List pdf files to be extracted
input_dir: str = "./data"
output_dir: str = "./output"
file_list: list[str] = os.listdir(input_dir)
client: OpenAI = OpenAI(api_key=json.load(open("./env.json"))["OPENAI_API_KEY"])
embedding_model:str = "text-embedding-3-small"
reasoning_model:str = "gpt-4o"
n:int = 1

In [ ]:
# Process each input file and write output to output directory
command: list[str]
completed_process: sp.CompletedProcess
scoring_df: pd.DataFrame
file_name: str
markdown_file: str
for file in file_list:
    command = [
        "marker_single",
        os.path.join(input_dir, file),
        output_dir,
        "--batch_multiplier",
        "1",
    ]
    print(f"Executing {' '.join(command)}")
    completed_process = sp.run(
        command, stderr=sp.STDOUT, stdout=sp.PIPE, text=True
    )

    if completed_process.returncode == 0:
        print(
            completed_process.stdout,
            completed_process.stderr,
        )
        print(f"Successfully processed {file}")
        file_name = file.split(".")[0]
        markdown_file = os.path.join(output_dir, f"{file_name}/{file_name}.md")
        print(f"Begin scoring report of {markdown_file} generated by {file}")
        scoring_df = scoring_markdown_annual_report(
            client=client,
            markdown_file_path=markdown_file,
            embedding_model=embedding_model,
            reasoning_model=reasoning_model,
            n=n,
        )
        scoring_df["input_file"] = file
    else:
        print(f"Failed to process {file}")
        print(
            completed_process.stdout,
            completed_process.stderr,
        )